# 02 — Build Dataset · Feature Engineering (3 Bases)

**Objetivo:** A partir das 3 bases temporais (TT / val / apl), aplicar feature engineering
uniforme e produzir bases ML-ready para os notebooks de splits e modelagem.

**Referência SPEC:** Seções 0.6, 4.1, 4.5, 4.6

---

| Entrada | Descrição |
|---|---|
| `data/gold/base_tt_raw.parquet` | Base treino/teste (Jan/2024–Mai/2025, com target) |
| `data/gold/base_val_raw.parquet` | Base validação temporal (Jun–Ago/2025, com target) |
| `data/gold/base_apl_raw.parquet` | Base aplicação (Set/2025+, sem target) |

| Saída | Descrição |
|---|---|
| `data/gold/base_features.parquet` | Camada 2: features + bins quintílicos (TT) |
| `data/gold/base_features_tt.parquet` | Camada 3: ML-ready — TT (OHE + ordinal + MICE) |
| `data/gold/base_features_val.parquet` | Camada 3: ML-ready — val (mesmos parâmetros de TT) |
| `data/gold/base_features_apl.parquet` | Camada 3: ML-ready — apl (mesmos parâmetros de TT) |
| `reports/dic_colunas_complemento.xlsx` | Dicionário de features para marcação de lag (ponto 15/15) |


## 1 · Setup & Configuração

In [29]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"pandas {pd.__version__}  ·  numpy {np.__version__}")
print(f"Execução: {datetime.now():%Y-%m-%d %H:%M}")

pandas 2.3.3  ·  numpy 2.3.5
Execução: 2026-04-30 15:08


In [30]:
# ════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ════════════════════════════════════════════════════════════════
PROJECT_ROOT   = Path.cwd().parent
DATA_GOLD      = PROJECT_ROOT / "data" / "gold"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS        = PROJECT_ROOT / "reports"

# Garantir pastas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# ── Grupos operacionais ───────────────────────────────────────
# O dataset unificado (pós MICE + binning) é divido por grupo ao final
# do notebook, gerando um parquet por grupo para os notebooks 03-05.
# Nota: MICE e binning são aplicados de forma unificada (toda a base).
# Isso é uma aproximação válida para dados de portfolio, pois os grupos
# compartilham a mesma escala de features e o MICE beneficia-se de mais dados.
GRUPOS    = ["Vendas", "Transporte", "Fábrica"]
COL_GRUPO = "ds_grupo"

# ── Parâmetros SPEC ───────────────────────────────────────────
JANELA_PREDICAO       = 4       # meses à frente (SPEC §2)
JANELA_TEMPORAL_MESES = 24      # últimos N meses (SPEC §4.1)
JANELA_SUBSAMPLING    = 4       # meses entre registros de ativos (SPEC §4.3)
THRESHOLD_GOWER_PERC  = 25      # percentil Gower para filtro similaridade (SPEC §4.3)
THRESHOLD_MIN_POS     = 80      # mínimo de positivos no treino (SPEC §2)
MIN_TEMPO_EMPRESA     = 3       # meses mínimos (regra dos 90 dias)

COL_TARGET  = f"fg_demitido_voluntario_{JANELA_PREDICAO}m"
COL_ID      = "id_colaborador"
COL_DATA    = "dt_mes_ano"

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print("Parâmetros SPEC:")
print(f"  Janela de predição    : {JANELA_PREDICAO} meses")
print(f"  Janela temporal       : {JANELA_TEMPORAL_MESES} meses")
print(f"  Janela subsampling    : {JANELA_SUBSAMPLING} meses")
print(f"  Percentil Gower       : {THRESHOLD_GOWER_PERC}")
print(f"  Min positivos treino  : {THRESHOLD_MIN_POS}")
print(f"  Min tempo empresa     : {MIN_TEMPO_EMPRESA} meses")
print(f"  Grupos operacionais   : {GRUPOS}")


Parâmetros SPEC:
  Janela de predição    : 4 meses
  Janela temporal       : 24 meses
  Janela subsampling    : 4 meses
  Percentil Gower       : 25
  Min positivos treino  : 80
  Min tempo empresa     : 3 meses
  Grupos operacionais   : ['Vendas', 'Transporte', 'Fábrica']


## 2 · Carga & Filtro Temporal (SPEC §4.1)

In [31]:
# ── Carregar base de treino/validação temporal (base_tt_raw) ─
df = pd.read_parquet(DATA_GOLD / "base_tt_raw.parquet")
df[COL_DATA] = pd.to_datetime(df[COL_DATA])
df[COL_ID]   = df[COL_ID].astype(str)

print(f"base_tt_raw carregada: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  IDs: {df[COL_ID].nunique():,}  |  Período: {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")


base_tt_raw carregada: 128,249 linhas × 461 colunas
  IDs: 11,351  |  Período: 2024-01 → 2025-05


In [32]:
# ── base_tt_raw já filtrada em 01_ingest (Jan/2024–Mai/2025) ─
# Filtro temporal e dropna do target foram aplicados em 01_ingest.ipynb.
# Nenhuma ação necessária aqui.
print(f"✓ base_tt_raw já filtrada por período e target em 01_ingest.ipynb")
print(f"  Período: {df[COL_DATA].min():%Y-%m} → {df[COL_DATA].max():%Y-%m}")
print(f"  Linhas : {len(df):,}  |  IDs: {df[COL_ID].nunique():,}")


✓ base_tt_raw já filtrada por período e target em 01_ingest.ipynb
  Período: 2024-01 → 2025-05
  Linhas : 128,249  |  IDs: 11,351


In [33]:
# ── Filtro de tempo de empresa ────────────────────────────────
# MOVIDO para 03_splits.ipynb — aplicado apenas nos splits do modelo,
# não nas bases gold. As bases gold mantêm todos os colaboradores.
print("ℹ Filtro vl_tempo_empresa > 3m aplicado em 03_splits.ipynb (não aqui).")
print(f"  Base TT mantida integral: {len(df):,} linhas | {df[COL_ID].nunique():,} IDs")


ℹ Filtro vl_tempo_empresa > 3m aplicado em 03_splits.ipynb (não aqui).
  Base TT mantida integral: 128,249 linhas | 11,351 IDs


## 3 · Seleção da Janela de Predição (SPEC §2)

Testar janela de 4 → 6 → 12 meses. Se nenhuma atinge o threshold de positivos, abortar.

In [34]:

# ── Seleção automática da janela de predição ──────────────────
# As colunas fg_demitido_voluntario_Km foram criadas em 01_ingest
JANELAS_CANDIDATAS = [4, 6, 12]

janela_selecionada = None
col_target_final   = None

for k in JANELAS_CANDIDATAS:
    col_k = f"fg_demitido_voluntario_{k}m"
    if col_k not in df.columns:
        print(f"  Janela {k}m: coluna {col_k} não encontrada — pulando")
        continue
    
    n_pos = (df[col_k] == 1).sum()
    n_tot = df[col_k].notna().sum()
    taxa  = n_pos / n_tot if n_tot > 0 else 0
    
    print(f"  Janela {k:>2d}m: {n_pos:>6,} positivos / {n_tot:>8,} válidos ({taxa:.2%})")
    
    if n_pos >= THRESHOLD_MIN_POS and janela_selecionada is None:
        janela_selecionada = k
        col_target_final   = col_k

if janela_selecionada is None:
    # Fallback: usar a coluna de target que já existe
    fallback_cols = [c for c in df.columns if c.startswith("fg_demitido_voluntario")]
    if fallback_cols:
        col_target_final   = fallback_cols[0]
        janela_selecionada = JANELA_PREDICAO
        n_pos = (df[col_target_final] == 1).sum()
        print(f"\n⚠ Nenhuma janela atingiu {THRESHOLD_MIN_POS} positivos.")
        print(f"  Usando fallback: {col_target_final} com {n_pos} positivos")
    else:
        raise ValueError("Nenhuma coluna de target encontrada. Verifique as bases.")
else:
    print(f"\n✓ Janela selecionada: {janela_selecionada} meses → {col_target_final}")

COL_TARGET = col_target_final
JANELA_PREDICAO = janela_selecionada


  Janela  4m: 17,986 positivos /  128,249 válidos (14.02%)
  Janela  6m: 24,417 positivos /  124,066 válidos (19.68%)
  Janela 12m: 36,735 positivos /   88,312 válidos (41.60%)

✓ Janela selecionada: 4 meses → fg_demitido_voluntario_4m


In [35]:

# ── Remover registros com target NaN ──────────────────────────
n_antes = len(df)
df = df.dropna(subset=[COL_TARGET]).copy()
df[COL_TARGET] = df[COL_TARGET].astype(int)

print(f"Remoção target NaN: {n_antes:,} → {len(df):,}")

# ── Filtrar apenas colaboradores ativos ───────────────────────
# Registros pós-demissão (fg_ativo=0) não devem entrar no treino —
# seriam data leakage implícito (a saída já aconteceu).
if "fg_ativo" in df.columns:
    n_antes_fg = len(df)
    df = df[df["fg_ativo"] == 1].copy().reset_index(drop=True)
    print(f"Filtro fg_ativo==1: {n_antes_fg:,} → {len(df):,}  (removidos: {n_antes_fg - len(df):,})")


Remoção target NaN: 128,249 → 128,249
Filtro fg_ativo==1: 128,249 → 97,867  (removidos: 30,382)


## 4 · Classificação de Features

Separar features em categorias para o tratamento diferenciado no protótipo dos desligados (SPEC §4.2):
- **Contínuas estáveis:** salário, tempo de empresa (→ último valor)
- **Contínuas comportamentais:** faltas, horas extras, avaliação (→ média + slope)
- **Categóricas:** cargo, unidade (→ último valor)

In [36]:

# ── Colunas de metadados (não são features) ───────────────────
META_COLS = {COL_ID, COL_DATA, COL_TARGET}
# Adicionar colunas de target alternativas
META_COLS.update(c for c in df.columns if c.startswith("fg_demitido_voluntario"))
# Flags de demissão/status/ativo nunca entram como feature (data leakage)
META_COLS.update(c for c in df.columns if c.startswith("fg_"))
# Adicionar grupo se existir
for c in ["grupo", "ds_grupo", "agrupamento_final"]:
    if c in df.columns:
        META_COLS.add(c)

FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

# ── Classificar por tipo ──────────────────────────────────────
cat_cols = []
num_cols = []
for c in FEATURE_COLS:
    if df[c].dtype == "object" or df[c].nunique() < 15:
        cat_cols.append(c)
    else:
        num_cols.append(c)

# ── Subclassificar numéricas ──────────────────────────────────
# Heurística: colunas que variam pouco dentro do mesmo indivíduo = estáveis
#             colunas com alta variação intra-indivíduo = comportamentais
KEYWORDS_ESTAVEIS = ["salario", "tempo_empresa", "idade", "pib", "desemprego",
                     "custo_vida", "base_pay", "economico"]

num_estaveis = []
num_comportamentais = []

for c in num_cols:
    if any(kw in c.lower() for kw in KEYWORDS_ESTAVEIS):
        num_estaveis.append(c)
    else:
        num_comportamentais.append(c)

print(f"Features classificadas ({len(FEATURE_COLS)} total):")
print(f"  Contínuas estáveis         : {len(num_estaveis)}")
print(f"  Contínuas comportamentais  : {len(num_comportamentais)}")
print(f"  Categóricas                : {len(cat_cols)}")
print(f"  Meta (excluídas)           : {len(META_COLS)}")


Features classificadas (444 total):
  Contínuas estáveis         : 71
  Contínuas comportamentais  : 343
  Categóricas                : 30
  Meta (excluídas)           : 17


In [37]:
# ── Listar features por tipo ──────────────────────────────────
print("Contínuas estáveis (→ último valor no protótipo):")
for c in num_estaveis:
    print(f"  {c}")

print(f"\nContínuas comportamentais (→ média + slope no protótipo):")
for c in num_comportamentais[:20]:
    print(f"  {c}")
if len(num_comportamentais) > 20:
    print(f"  ... e mais {len(num_comportamentais)-20}")

print(f"\nCategóricas (→ último valor no protótipo):")
for c in cat_cols:
    print(f"  {c}  ({df[c].nunique()} categorias)")

Contínuas estáveis (→ último valor no protótipo):
  vl_salario
  vl_economico_taxa_desemprego
  vl_economico_custo_vida
  vl_economico_salario_medio
  vl_economico_turnover_cnae_regiao
  vl_economico_pib_per_capita
  vl_tempo_empresa
  vl_idade
  vl_salario_mean_3m
  vl_economico_taxa_desemprego_mean_3m
  vl_economico_custo_vida_mean_3m
  vl_economico_salario_medio_mean_3m
  vl_economico_turnover_cnae_regiao_mean_3m
  vl_economico_pib_per_capita_mean_3m
  vl_tempo_empresa_mean_3m
  vl_idade_mean_3m
  vl_salario_min_3m
  vl_economico_taxa_desemprego_min_3m
  vl_economico_custo_vida_min_3m
  vl_economico_salario_medio_min_3m
  vl_economico_turnover_cnae_regiao_min_3m
  vl_economico_pib_per_capita_min_3m
  vl_tempo_empresa_min_3m
  vl_idade_min_3m
  vl_salario_max_3m
  vl_economico_taxa_desemprego_max_3m
  vl_economico_custo_vida_max_3m
  vl_economico_salario_medio_max_3m
  vl_economico_turnover_cnae_regiao_max_3m
  vl_economico_pib_per_capita_max_3m
  vl_tempo_empresa_max_3m
  vl_idade_m

## 4.5 · Categorização de Variáveis Contínuas/Ordinais

Variáveis numéricas contínuas ou ordinais com mais de 10 valores distintos são discretizadas
em até **5 categorias ordinais**, seguindo um método **não-paramétrico baseado em quintis**
(percentis amostrais, sem assumir distribuição normal ou outra forma específica).

### Regras de categorização

| Situação | Tratamento |
|---|---|
| Valor **nulo** | Mantido como `NaN` — não imputado nesta etapa |
| Valor **exatamente zero** | Categoria própria `"1_zero"` (separa zeros estruturais de valores contínuos) |
| Valores **não-nulos e não-zero** | Divididos em até 4 bins por quintis (se há zeros) ou 5 bins (se não há zeros) |
| **Distribuição muito concentrada** (cortes duplicados) | Bins duplicados são eliminados automaticamente — resultado pode ter < 5 categorias |

As categorias são nomeadas com o **número ordinal no início** seguido do **intervalo de corte**:
`"2_(-inf, 1.3]"`, `"3_(1.3, 5.7]"`, etc. Isso permite transformação ordinal direta por ordenação alfabética.

> Cada variável categorizada gera uma coluna `{nome_original}_bin` no dataset.
> As colunas originais são mantidas para análise comparativa.

### Saída desta etapa

`data/gold/base_features.parquet` — base analítica **com novas features**,
pronta para as próximas etapas de codificação (OHE) e preparação final para o modelo.


In [38]:
def criar_bins_quintil(series, n_bins_max=5):
    """
    Categorização quintílica não-paramétrica de uma série numérica.

    Algoritmo:
    ----------
    1. NaN → mantidos como NaN (sem imputação)
    2. Zero exato → categoria "1_zero" (isolado do resto da distribuição)
    3. Valores não-zero e não-nulos → divididos em até (n_bins_max - 1) bins
       se há zeros, ou n_bins_max bins caso contrário.
    4. Quantis calculados sobre os valores não-zero da distribuição real (não-paramétrico).
    5. Se cortes duplicados → eliminados automaticamente (bins < n_bins_max).
    6. Categoria nomeada: "{rank}_{(lo, hi]}" para rastreabilidade.

    Por que não-paramétrico?
    Ao usar percentis amostrais (ao invés de mean ± std), o método é robusto a
    assimetrias, caudas pesadas e distribuições multimodais — comuns em dados de RH.
    """
    s = series.copy()
    result = pd.Series(np.nan, index=s.index, dtype=object)

    has_zero     = (s == 0).any()
    mask_nonzero = s.notna() & (s != 0)
    vals_nz      = s[mask_nonzero]

    if len(vals_nz) == 0:
        if has_zero:
            result[s == 0] = "1_zero"
        return result

    # ── Calcular cortes por quintis (sem duplicatas) ──────────────
    n_bins   = (n_bins_max - 1) if has_zero else n_bins_max
    quantis  = np.linspace(0, 1, n_bins + 1)
    cortes   = np.unique(np.quantile(vals_nz.dropna(), quantis))

    # Se muito poucos cortes únicos, usar mínimo de 1 bin real
    if len(cortes) < 2:
        cortes = np.array([vals_nz.min(), vals_nz.max()])

    # Substituir extremos por -inf/+inf para cobrir toda a distribuição
    cortes[0]  = -np.inf
    cortes[-1] =  np.inf
    actual_bins = len(cortes) - 1

    # ── Montar labels ordinais ────────────────────────────────────
    rank_start = 2 if has_zero else 1
    labels = []
    for i in range(actual_bins):
        lo = cortes[i]
        hi = cortes[i + 1]
        lo_str = "−∞"    if np.isinf(lo) and lo < 0 else f"{lo:.2f}"
        hi_str = "+∞"    if np.isinf(hi) and hi > 0 else f"{hi:.2f}"
        labels.append(f"{rank_start + i}_({lo_str},{hi_str}]")

    # ── Aplicar pd.cut ────────────────────────────────────────────
    try:
        binned = pd.cut(
            vals_nz,
            bins=cortes,
            labels=labels,
            include_lowest=True,
            right=True,
        )
        result[mask_nonzero] = binned.astype(object)
    except Exception as e:
        # Fallback: classificação com rank percentil
        ranked = pd.qcut(vals_nz.rank(method="first"), q=min(actual_bins, 4),
                         labels=False, duplicates="drop")
        result[mask_nonzero] = (ranked + rank_start).astype(str).apply(lambda x: f"{x}_outros")

    # ── Categoria de zero ─────────────────────────────────────────
    if has_zero:
        result[s == 0] = "1_zero"

    # NaN permanece NaN
    result[s.isna()] = np.nan
    return result


print("Função criar_bins_quintil definida ✓")


Função criar_bins_quintil definida ✓


In [39]:
# ── Selecionar colunas candidatas à categorização ────────────
# Critérios: numéricas (float64 ou int64), não são meta, > 10 valores distintos
NUM_DTYPES = ["float64", "int64", "float32", "int32"]

cols_para_bin = [
    c for c in df.columns
    if c not in META_COLS
    and str(df[c].dtype) in NUM_DTYPES
    and df[c].nunique(dropna=True) > 10
]

print(f"Colunas candidatas à categorização: {len(cols_para_bin)}")
for c in cols_para_bin:
    has_zero = (df[c] == 0).any()
    n_null   = df[c].isna().sum()
    print(f"  {c:<50s} nunique={df[c].nunique():>5}  zeros={'sim' if has_zero else 'não':3}  nulos={n_null}")


Colunas candidatas à categorização: 414
  vl_tempo_funcao                                    nunique=   35  zeros=sim  nulos=0
  vl_salario                                         nunique=97820  zeros=não  nulos=0
  vl_beneficio_tempo_1                               nunique=19422  zeros=sim  nulos=0
  vl_beneficio_tempo_2                               nunique=25316  zeros=sim  nulos=0
  vl_beneficio_extra_1                               nunique=28927  zeros=sim  nulos=0
  vl_beneficio_extra_2                               nunique=78668  zeros=sim  nulos=0
  vl_beneficio_extra_3                               nunique=11098  zeros=sim  nulos=0
  vl_beneficio_extra_4                               nunique=78258  zeros=sim  nulos=0
  vl_remuneracao_variavel                            nunique=31966  zeros=sim  nulos=0
  vl_beneficio_extra_5                               nunique= 6267  zeros=sim  nulos=0
  vl_beneficio_extra_10                              nunique=22415  zeros=sim  nulos=0
  v

In [40]:
# ── Aplicar categorização e gerar colunas _bin ───────────────
bin_cols_criadas = []
bin_resumo       = []
bin_edges_dict   = {}   # cortes de TT → reutilizados em val/apl

for col in cols_para_bin:
    col_bin = f"{col}_bin"
    df[col_bin] = criar_bins_quintil(df[col])
    bin_cols_criadas.append(col_bin)

    # Verificação: quantas categorias geradas
    cats = df[col_bin].dropna().unique()
    n_cats = len(cats)
    n_null = df[col_bin].isna().sum()
    bin_resumo.append({
        "coluna_original": col,
        "coluna_bin":      col_bin,
        "n_categorias":    n_cats,
        "nulos":           n_null,
    })

    # ── Salvar cortes para val/apl ────────────────────────────
    s        = df[col]
    has_zero = bool((s == 0).any())
    mask_nz  = s.notna() & (s != 0)
    vals_nz  = s[mask_nz].dropna()
    if len(vals_nz) > 0:
        n_bins  = (5 - 1) if has_zero else 5
        quantis = np.linspace(0, 1, n_bins + 1)
        cortes  = np.unique(np.quantile(vals_nz, quantis)).astype(float)
        if len(cortes) < 2:
            cortes = np.array([float(vals_nz.min()), float(vals_nz.max())])
        cortes[0]  = -np.inf
        cortes[-1] = +np.inf
        rank_start  = 2 if has_zero else 1
        actual_bins = len(cortes) - 1
        labels = []
        for i in range(actual_bins):
            lo, hi = cortes[i], cortes[i + 1]
            lo_str = "−∞" if np.isinf(lo) and lo < 0 else f"{lo:.2f}"
            hi_str = "+∞" if np.isinf(hi) and hi > 0 else f"{hi:.2f}"
            labels.append(f"{rank_start + i}_({lo_str},{hi_str}]")
        bin_edges_dict[col] = {
            "has_zero": has_zero, "cortes": cortes, "labels": labels
        }

print(f"✓ {len(bin_cols_criadas)} colunas _bin criadas")
print(f"✓ bin_edges_dict salvo para {len(bin_edges_dict)} colunas (reusar em val/apl)\n")

resumo_df = pd.DataFrame(bin_resumo).set_index("coluna_original")
print(resumo_df.to_string())
print(f"\nShape do df após bins: {df.shape[0]:,} linhas × {df.shape[1]} colunas")


C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_bin] = criar_bins_quintil(df[col])
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_bin] = criar_bins_quintil(df[col])
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

✓ 414 colunas _bin criadas
✓ bin_edges_dict salvo para 414 colunas (reusar em val/apl)

                                                                                      coluna_bin  n_categorias  nulos
coluna_original                                                                                                      
vl_tempo_funcao                                                              vl_tempo_funcao_bin             4      0
vl_salario                                                                        vl_salario_bin             5      0
vl_beneficio_tempo_1                                                    vl_beneficio_tempo_1_bin             5      0
vl_beneficio_tempo_2                                                    vl_beneficio_tempo_2_bin             5      0
vl_beneficio_extra_1                                                    vl_beneficio_extra_1_bin             5      0
vl_beneficio_extra_2                                                    vl_beneficio_e

C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_bin] = criar_bins_quintil(df[col])
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_bin] = criar_bins_quintil(df[col])
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2992027819.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

In [41]:
# ── Salvar base com novas features ───────────────────────────
# Versão 2 da base: input + indicadores de grupo + variáveis binadas
base_features_path = DATA_GOLD / "base_features.parquet"
df.to_parquet(base_features_path, index=False)

mb = base_features_path.stat().st_size / 1e6
print(f"✓ base_features.parquet salva")
print(f"  Shape   : {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"  Colunas _bin adicionadas : {len(bin_cols_criadas)}")
print(f"  Colunas originais mantidas: {df.shape[1] - len(bin_cols_criadas)}")
print(f"  Tamanho : {mb:.1f} MB")
print(f"  Salvo em: {base_features_path}")
print()
print("─── Exemplo: distribuição de uma coluna _bin ─────────────")
if bin_cols_criadas:
    ex_col = bin_cols_criadas[0]
    vc = df[ex_col].value_counts(dropna=False).sort_index()
    print(f"  {ex_col}:")
    print(vc.to_string())


✓ base_features.parquet salva
  Shape   : 97,867 linhas × 875 colunas
  Colunas _bin adicionadas : 414
  Colunas originais mantidas: 461
  Tamanho : 128.0 MB
  Salvo em: g:\Meu Drive\Rafael\Desemprego\projetos github\clonados\projeto1_modelo_preditivo\data\gold\base_features.parquet

─── Exemplo: distribuição de uma coluna _bin ─────────────
  vl_tempo_funcao_bin:
vl_tempo_funcao_bin
1_zero           30667
2_(−∞,2.00]      38140
3_(2.00,4.00]    13628
4_(4.00,+∞]      15432


In [42]:
# ════════════════════════════════════════════════════════════════
# DICIONÁRIO DE COLUNAS — dic_colunas_complemento.xlsx
# Salva a lista completa de features para que o usuário possa
# marcar lag=1 nas colunas de ponto (apuração 15/15).
#
# ⚠ AÇÃO NECESSÁRIA (primeira execução):
#   1. Abra reports/dic_colunas_complemento.xlsx
#   2. Marque lag=1 nas colunas cujo ponto tem apuração 15/15
#   3. Re-execute A PARTIR DESTA CÉLULA (inclusive) para propagar a correção
# ════════════════════════════════════════════════════════════════
dict_path = REPORTS / "dic_colunas_complemento.xlsx"

# Features disponíveis neste ponto (base_features: originais + bins)
all_features_now = [c for c in df.columns if c not in META_COLS]

if not dict_path.exists():
    dic_df = pd.DataFrame({
        "Feature":   all_features_now,
        "lag":       0,
        "descricao": ""
    })
    dic_df.to_excel(dict_path, index=False)
    print(f"✓ Criado: dic_colunas_complemento.xlsx  ({len(all_features_now)} features)")
    print(f"  Em: {dict_path}")
    print("\n⚠  AÇÃO: abra o arquivo, marque lag=1 para colunas de ponto (apuração 15/15)")
    print("   e re-execute a partir desta célula.")
else:
    dic_df = pd.read_excel(dict_path)
    existing = set(dic_df["Feature"].tolist())
    new_feats = [c for c in all_features_now if c not in existing]

    if new_feats:
        new_rows = pd.DataFrame({"Feature": new_feats, "lag": 0, "descricao": ""})
        dic_df   = pd.concat([dic_df, new_rows], ignore_index=True)
        dic_df.to_excel(dict_path, index=False)
        print(f"✓ dic_colunas_complemento.xlsx atualizado: +{len(new_feats)} novas features")
    else:
        print(f"✓ dic_colunas_complemento.xlsx já existe e atualizado")

    n_lag = int((dic_df.get("lag", pd.Series(dtype=int)) == 1).sum())
    print(f"  Features totais: {len(dic_df)}  |  com lag=1: {n_lag}")


✓ dic_colunas_complemento.xlsx já existe e atualizado
  Features totais: 1078  |  com lag=1: 0


In [43]:
# ════════════════════════════════════════════════════════════════
# CORREÇÃO DE LAG — colunas de ponto (apuração 15/15)
# Para colunas marcadas com lag=1: usa o valor do mês anterior
# dentro do mesmo colaborador (shift+1 no grupo de ID).
# Primeiro registro de cada ID: mantém valor original (sem mês anterior).
# ════════════════════════════════════════════════════════════════
dict_path = REPORTS / "dic_colunas_complemento.xlsx"

lag_cols = []
if dict_path.exists():
    dic_df   = pd.read_excel(dict_path)
    if "lag" in dic_df.columns:
        lag_cols = dic_df[dic_df["lag"] == 1]["Feature"].tolist()
        lag_cols = [c for c in lag_cols if c in df.columns]

if lag_cols:
    print(f"⚠  Aplicando lag-1 em {len(lag_cols)} colunas de ponto:")
    df = df.sort_values([COL_ID, COL_DATA]).copy()
    for c in lag_cols:
        shifted     = df.groupby(COL_ID)[c].shift(1)
        df[c]       = shifted.fillna(df[c])   # 1º registro: mantém valor original
        print(f"  {c}")
    print(f"\n✓ Lag correction aplicado — §4.6 irá usar dados corrigidos")
else:
    print("✓ Nenhuma coluna com lag=1 — sem correção de lag necessária")
    if not dict_path.exists():
        print("  (Execute a célula acima para criar dic_colunas_complemento.xlsx)")


✓ Nenhuma coluna com lag=1 — sem correção de lag necessária


## 4.6 · Preparação Final para Modelagem (`base_features_tt`)

Transforma `base_features` em uma base **completamente ML-ready** — camada 3 do pipeline.
Todos os requisitos clássicos de ciência de dados são atendidos: **sem nulos**, **tudo numérico**.

| Transformação | Detalhe |
|---|---|
| **Seleção do target** | Mantém apenas `fg_demitido_voluntario_{K}m` selecionada em §3; remove as demais |
| **Ordinais** | `_bin` strings (`"1_zero"`, `"2_(14,27]"`) → inteiros ordinais (`1.0`, `2.0`, `3.0`…) usando o prefixo que definimos na categorização |
| **Conversão numérica** | Colunas `object` com valores numéricos (string) → `float64` |
| **One-Hot Encoding** | Categóricas com ≤ 5 categorias distintas → dummies binárias `0/1` |
| **Drop alta cardinalidade** | Categóricas com > 5 categorias → removidas (informação capturada por rolling stats ou não recuperável sem OHE massivo) |
| **Nulos > 20%** | Coluna removida |
| **Nulos ≤ 20%** | Imputação por **regressão iterativa (MICE)** — `IterativeImputer` do scikit-learn |

> O target **não** é incluído nos preditores da regressão de imputação: incluí-lo
> introduziria informação futura nos dados de treino (*data leakage*).

**Saída:** `data/gold/base_features_tt.parquet`


In [44]:

# ════════════════════════════════════════════════════════════════
# STEP 1 — Criar df_modelo e manter apenas o target selecionado
# ════════════════════════════════════════════════════════════════
df_modelo = df.copy()

# Remover todas as colunas de target exceto a janela selecionada em §3
outros_targets = [
    c for c in df_modelo.columns
    if c.startswith("fg_demitido_voluntario") and c != COL_TARGET
]
df_modelo.drop(columns=outros_targets, inplace=True)
print(f"Targets alternativos removidos: {len(outros_targets)}  →  mantido: '{COL_TARGET}'")

# Remover todas as flags fg_* — nunca devem ser features (data leakage).
# COL_TARGET começa com "fg_demitido_voluntario_" — é preservado.
fg_drop = [c for c in df_modelo.columns if c.startswith("fg_") and c != COL_TARGET]
df_modelo.drop(columns=fg_drop, inplace=True, errors="ignore")
print(f"Flags fg_* removidas (leakage prevention): {fg_drop}")

# Garantir sem nulos no target (dupla verificação — §3 já fez isso)
n_antes = len(df_modelo)
df_modelo.dropna(subset=[COL_TARGET], inplace=True)
print(f"Linhas com target nulo: {n_antes - len(df_modelo)} removidas")

print(f"\ndf_modelo: {df_modelo.shape[0]:,} linhas × {df_modelo.shape[1]} colunas")
vc_target = df_modelo[COL_TARGET].value_counts().sort_index().to_dict()
print(f"Distribuição target: {vc_target}  (taxa: {df_modelo[COL_TARGET].mean():.2%})")


Targets alternativos removidos: 9  →  mantido: 'fg_demitido_voluntario_4m'
Flags fg_* removidas (leakage prevention): ['fg_ativo', 'fg_dem_vol', 'fg_dem_inv', 'fg_sem_output']
Linhas com target nulo: 0 removidas

df_modelo: 97,867 linhas × 862 colunas
Distribuição target: {0: 81258, 1: 16609}  (taxa: 16.97%)


In [45]:
# ════════════════════════════════════════════════════════════════
# STEP 2 — Ordinal encoding das colunas _bin
# Converte strings "1_zero", "2_(14,27]" → inteiro ordinal 1.0, 2.0 ...
# usando o prefixo numérico que foi colocado propositalmente na categorização §4.5
# ════════════════════════════════════════════════════════════════

def extrair_ordinal_bin(valor):
    """
    Extrai o prefixo ordinal da string de categoria.
    '3_(27.00,53.00]'  →  3.0
    '1_zero'           →  1.0
    NaN                →  NaN
    """
    if pd.isna(valor):
        return np.nan
    try:
        return float(str(valor).split("_")[0])
    except (ValueError, IndexError):
        return np.nan


bin_cols_no_modelo = [c for c in bin_cols_criadas if c in df_modelo.columns]

for col in bin_cols_no_modelo:
    df_modelo[col] = df_modelo[col].map(extrair_ordinal_bin)

print(f"✓ Ordinal encoding: {len(bin_cols_no_modelo)} colunas _bin → float ordinal")
print("  Antes: strings tipo '2_(14.00,27.00]'  →  Depois: 2.0")
print("  NaN mantidos como NaN\n")

if bin_cols_no_modelo:
    ex = bin_cols_no_modelo[0]
    vc = df_modelo[ex].value_counts(dropna=False).sort_index()
    print(f"Exemplo — {ex}:")
    print(vc.to_string())


✓ Ordinal encoding: 414 colunas _bin → float ordinal
  Antes: strings tipo '2_(14.00,27.00]'  →  Depois: 2.0
  NaN mantidos como NaN

Exemplo — vl_tempo_funcao_bin:
vl_tempo_funcao_bin
1.0    30667
2.0    38140
3.0    13628
4.0    15432


In [46]:
# ════════════════════════════════════════════════════════════════
# STEP 3 — Conversão numérica + OHE + drop alta cardinalidade
# ════════════════════════════════════════════════════════════════
# COL_GRUPO é metadado de agrupamento — não entra no OHE nem nas features;
# precisa ser preservado até o final para o split por grupo em #VSC-1aaf8e8b.
META_PRESERVE = {COL_ID, COL_DATA, COL_GRUPO}   # identificadores + grupo — não alterar

# Colunas object que não são identificadores
obj_cols = [
    c for c in df_modelo.columns
    if df_modelo[c].dtype == object and c not in META_PRESERVE
]

# ── Tentar converter object → float64 ────────────────────────
# Aceita a conversão se introduziu < 10% de novos nulos
converted_to_num = []
still_obj        = []

for c in obj_cols:
    null_antes  = df_modelo[c].isna().mean()
    conv        = pd.to_numeric(df_modelo[c], errors="coerce")
    null_depois = conv.isna().mean()
    if null_depois - null_antes < 0.10:
        df_modelo[c] = conv
        converted_to_num.append(c)
    else:
        still_obj.append(c)

print(f"Colunas object identificadas : {len(obj_cols)}")
print(f"  Convertidas para float64   : {len(converted_to_num)}")
print(f"  Ainda categóricas (texto)  : {len(still_obj)}")

# ── Split por cardinalidade: OHE ≤5, drop >5 ─────────────────
ohe_cols  = [c for c in still_obj if df_modelo[c].nunique(dropna=True) <= 5]
drop_cols = [c for c in still_obj if df_modelo[c].nunique(dropna=True) > 5]

print(f"\nCategóricas com ≤ 5 distintos (→ OHE)  : {len(ohe_cols)}")
for c in ohe_cols:
    cats_list = sorted(df_modelo[c].dropna().unique().tolist())
    print(f"  {c}: {cats_list}")

print(f"\nCategóricas com > 5 distintos (→ drop)  : {len(drop_cols)}")
for c in drop_cols:
    print(f"  {c}: {df_modelo[c].nunique()} categorias")

# ── Aplicar OHE ──────────────────────────────────────────────
if ohe_cols:
    dummies        = pd.get_dummies(df_modelo[ohe_cols], dummy_na=False, dtype=float)
    ohe_dummy_cols = list(dummies.columns)
    df_modelo      = pd.concat([df_modelo.drop(columns=ohe_cols), dummies], axis=1)
    print(f"\n✓ OHE: {len(ohe_cols)} colunas originais → {len(dummies.columns)} binárias")
    print(f"  Novas colunas: {ohe_dummy_cols}")
else:
    ohe_dummy_cols = []
    print("\n⚠ Nenhuma coluna para OHE")

# ── Drop high-cardinality ─────────────────────────────────────
if drop_cols:
    df_modelo.drop(columns=drop_cols, inplace=True)
    print(f"✓ {len(drop_cols)} colunas de alta cardinalidade removidas")

print(f"\nShape após encoding: {df_modelo.shape[0]:,} × {df_modelo.shape[1]} colunas")


Colunas object identificadas : 15
  Convertidas para float64   : 1
  Ainda categóricas (texto)  : 14

Categóricas com ≤ 5 distintos (→ OHE)  : 5
  ds_genero: ['F', 'M']
  ds_pcd_detalhe: ['Não', 'Sim']
  ds_tipo_turno: ['Diurno_5x2', 'Diurno_6x1', 'Revezamento_12x36']
  ds_segmento: ['Operação']
  ds_agrupamento: ['Operação - Fábrica', 'Operação - Transporte', 'Operação - Vendas']

Categóricas com > 5 distintos (→ drop)  : 9
  ds_cargo: 101 categorias
  ds_uorg: 88 categorias
  ds_centro_custo: 727 categorias
  ds_etnia: 6 categorias
  ds_pcd: 7 categorias
  ds_escolaridade: 12 categorias
  ds_nivel_cargo: 8 categorias
  ds_gestor: 1428 categorias
  ds_estado: 17 categorias

✓ OHE: 5 colunas originais → 11 binárias
  Novas colunas: ['ds_genero_F', 'ds_genero_M', 'ds_pcd_detalhe_Não', 'ds_pcd_detalhe_Sim', 'ds_tipo_turno_Diurno_5x2', 'ds_tipo_turno_Diurno_6x1', 'ds_tipo_turno_Revezamento_12x36', 'ds_segmento_Operação', 'ds_agrupamento_Operação - Fábrica', 'ds_agrupamento_Operação - Tran

In [47]:

# ════════════════════════════════════════════════════════════════
# STEP 4 — Tratamento de nulos
# Colunas com > 20% nulos → removidas
# Colunas com ≤ 20% nulos → imputação por mediana (por coluna)
# O target NÃO entra como preditor na imputação
# Nota: MICE (IterativeImputer) foi substituído por SimpleImputer(median)
# pois com ~860 colunas e valores de grande escala, MICE gera overflow
# numérico no BayesianRidge interno. Mediana é robusta e suficiente aqui.
# ════════════════════════════════════════════════════════════════
from sklearn.impute import SimpleImputer

# Colunas de features numéricas para avaliar/imputar (sem meta e sem target)
feat_num_cols = [
    c for c in df_modelo.columns
    if c not in {COL_ID, COL_DATA, COL_TARGET}
    and pd.api.types.is_numeric_dtype(df_modelo[c])
]

# ── Verificar nulos por coluna ─────────────────────────────────
null_series = df_modelo[feat_num_cols].isna().mean().sort_values(ascending=False)
high_null      = null_series[null_series > 0.20]
high_null_cols = high_null.index.tolist()
low_null       = null_series[(null_series > 0) & (null_series <= 0.20)]

print(f"Total features numéricas: {len(feat_num_cols)}")

if len(high_null):
    print(f"\nColunas com >20% nulos → REMOVIDAS ({len(high_null)}):")
    for c, p in high_null.items():
        print(f"  {c:<55s} {p:.1%}")
else:
    print("\n✓ Nenhuma coluna com >20% de nulos")

if len(low_null):
    print(f"\nColunas com ≤20% nulos → IMPUTADAS ({len(low_null)}):")
    for c, p in low_null.items():
        print(f"  {c:<55s} {p:.1%}")
else:
    print("✓ Nenhuma coluna com nulos a imputar")

# ── Remover colunas com >20% nulos ────────────────────────────
df_modelo.drop(columns=high_null.index.tolist(), inplace=True)
feat_num_cols = [c for c in feat_num_cols if c not in high_null.index]

print(f"\nShape após remoção: {df_modelo.shape[0]:,} × {df_modelo.shape[1]}")

# ── Imputação por mediana ──────────────────────────────────────
imp = None   # inicializar; reutilizado em _transform_partition
null_total = int(df_modelo[feat_num_cols].isna().sum().sum())
if null_total > 0:
    print(f"\nImputando {null_total:,} nulos restantes...")
    print("  Método: SimpleImputer(strategy='median') — target excluído dos preditores")
    imp = SimpleImputer(strategy="median")
    df_modelo[feat_num_cols] = imp.fit_transform(df_modelo[feat_num_cols])
    null_restante = int(df_modelo[feat_num_cols].isna().sum().sum())
    print(f"  ✓ Nulos restantes: {null_restante}")
else:
    print("✓ Sem nulos nas features — imputação não necessária")

print(f"\nNulos totais finais: {int(df_modelo.drop(columns=[COL_ID, COL_DATA]).isna().sum().sum())}")


Total features numéricas: 855

✓ Nenhuma coluna com >20% de nulos

Colunas com ≤20% nulos → IMPUTADAS (229):
  vl_abs_dias_sum_12m_std_6m_bin                          13.7%
  vl_abs_dias_sum_6m_std_6m_bin                           13.7%
  vl_abs_dias_sum_3m_std_6m_bin                           13.7%
  vl_abs_dias_sum_12m_std_3m_bin                          13.7%
  vl_abs_dias_sum_6m_std_3m_bin                           13.7%
  vl_abs_dias_sum_3m_std_3m_bin                           13.7%
  vl_abs_dias_sum_3m_std_6m                               13.7%
  vl_abs_dias_sum_6m_std_3m                               13.7%
  vl_abs_dias_sum_6m_std_6m                               13.7%
  vl_abs_dias_sum_12m_std_3m                              13.7%
  vl_abs_dias_sum_3m_std_3m                               13.7%
  vl_abs_dias_sum_12m_std_6m                              13.7%
  vl_tempo_empresa_std_6m_bin                             11.1%
  vl_distancia_trabalho_min_std_6m_bin                    1

In [48]:
# ── Salvar base_features_tt.parquet — camada 3 do pipeline ───
base_features_tt_path = DATA_GOLD / "base_features_tt.parquet"
df_modelo.to_parquet(base_features_tt_path, index=False)

mb = base_features_tt_path.stat().st_size / 1e6
print(f"✓ base_features_tt.parquet salva  —  camada 3 do pipeline")
print(f"  Shape       : {df_modelo.shape[0]:,} linhas × {df_modelo.shape[1]} colunas")
print(f"  IDs únicos  : {df_modelo[COL_ID].nunique():,}")
print(f"  Nulos totais: {int(df_modelo.isna().sum().sum())}")
print(f"  Tamanho     : {mb:.1f} MB")
print(f"  Target      : {COL_TARGET}")
dist = dict(df_modelo[COL_TARGET].value_counts().sort_index())
print(f"  Target dist : {dist}  (taxa: {df_modelo[COL_TARGET].mean():.2%})")


✓ base_features_tt.parquet salva  —  camada 3 do pipeline
  Shape       : 97,867 linhas × 859 colunas
  IDs únicos  : 10,897
  Nulos totais: 0
  Tamanho     : 126.8 MB
  Target      : fg_demitido_voluntario_4m
  Target dist : {0: np.int64(81258), 1: np.int64(16609)}  (taxa: 16.97%)


In [49]:

# ════════════════════════════════════════════════════════════════
# STEP 5 — Transformar base_val_raw e base_apl_raw com os mesmos
#           parâmetros do pipeline TT (bins, OHE, imputação)
#
# Parâmetros reutilizados de TT:
#   bin_edges_dict  — cortes quintílicos por coluna
#   ohe_cols        — colunas categóricas ≤5 distintos (antes do OHE)
#   ohe_dummy_cols  — colunas dummy criadas no OHE de TT
#   drop_cols       — colunas de alta cardinalidade removidas em TT
#   high_null_cols  — colunas removidas por >20% nulos em TT
#   feat_num_cols   — features numéricas para imputação em TT
#   imp             — SimpleImputer ajustado em TT (ou None)
# ════════════════════════════════════════════════════════════════

def _transform_partition(df_raw, partition_name):
    """Aplica o pipeline de feature engineering de TT em uma base nova."""
    df_t = df_raw.copy()
    df_t[COL_DATA] = pd.to_datetime(df_t[COL_DATA])
    df_t[COL_ID]   = df_t[COL_ID].astype(str)

    # ── Filtro fg_ativo == 1 ──────────────────────────────────
    # Idêntico ao filtro da base_tt: apenas snapshots de colaboradores
    # ativos devem ser usados para treino, validação e scoring.
    if "fg_ativo" in df_t.columns:
        df_t = df_t[df_t["fg_ativo"] == 1].copy().reset_index(drop=True)

    # ── Remover targets alternativos ──────────────────────────
    outros = [c for c in df_t.columns
              if c.startswith("fg_demitido_voluntario") and c != COL_TARGET]
    df_t.drop(columns=outros, inplace=True, errors="ignore")

    # ── Remover flags fg_* (exceto target) ────────────────────
    # Flags de demissão, fg_ativo, fg_sem_output etc nunca são features.
    fg_drop_t = [c for c in df_t.columns if c.startswith("fg_") and c != COL_TARGET]
    df_t.drop(columns=fg_drop_t, inplace=True, errors="ignore")

    # ── Bins: aplicar cortes salvos de TT ─────────────────────
    for col, info in bin_edges_dict.items():
        if col not in df_t.columns:
            continue
        s      = df_t[col].copy()
        result = pd.Series(np.nan, index=s.index, dtype=object)
        if info["has_zero"]:
            result[s == 0] = "1_zero"
        mask_nz = s.notna() & (s != 0)
        if mask_nz.any():
            try:
                binned = pd.cut(s[mask_nz], bins=info["cortes"],
                                labels=info["labels"], include_lowest=True, right=True)
                result[mask_nz] = binned.astype(object)
            except Exception:
                pass
        result[s.isna()] = np.nan
        df_t[f"{col}_bin"] = result

    # ── Ordinal encoding dos _bin ─────────────────────────────
    for col in [c for c in df_t.columns if c.endswith("_bin")]:
        df_t[col] = df_t[col].map(extrair_ordinal_bin)

    # ── Conversão numérica de colunas object (não-OHE) ───────
    obj_non_ohe = [
        c for c in df_t.columns
        if df_t[c].dtype == object
        and c not in {COL_ID, COL_DATA, COL_GRUPO}
        and c not in ohe_cols
    ]
    for c in obj_non_ohe:
        null_antes = df_t[c].isna().mean()
        conv = pd.to_numeric(df_t[c], errors="coerce")
        if conv.isna().mean() - null_antes < 0.10:
            df_t[c] = conv

    # ── Drop alta cardinalidade ───────────────────────────────
    df_t.drop(columns=[c for c in drop_cols if c in df_t.columns], inplace=True)

    # ── OHE alinhado ao schema de TT ─────────────────────────
    ohe_cols_present = [c for c in ohe_cols if c in df_t.columns]
    if ohe_cols_present:
        dummies_t = pd.get_dummies(df_t[ohe_cols_present], dummy_na=False, dtype=float)
        dummies_t = dummies_t.reindex(columns=ohe_dummy_cols, fill_value=0)
        df_t = pd.concat([df_t.drop(columns=ohe_cols_present), dummies_t], axis=1)

    # ── Drop colunas com >20% nulos (removidas em TT) ────────
    df_t.drop(columns=[c for c in high_null_cols if c in df_t.columns], inplace=True)

    # ── Imputação (transform com SimpleImputer ajustado em TT) ──
    feat_num_t = [c for c in feat_num_cols if c in df_t.columns]
    if imp is not None and df_t[feat_num_t].isna().sum().sum() > 0:
        df_t[feat_num_t] = imp.transform(df_t[feat_num_t])

    return df_t


print("Transformando base_val_raw e base_apl_raw com parâmetros de TT...")

base_val_raw = pd.read_parquet(DATA_GOLD / "base_val_raw.parquet")
base_apl_raw = pd.read_parquet(DATA_GOLD / "base_apl_raw.parquet")

df_val = _transform_partition(base_val_raw, "val")
df_apl = _transform_partition(base_apl_raw, "apl")

# ── Salvar ───────────────────────────────────────────────────
path_val = DATA_GOLD / "base_features_val.parquet"
path_apl = DATA_GOLD / "base_features_apl.parquet"

df_val.to_parquet(path_val, index=False)
df_apl.to_parquet(path_apl, index=False)

# ── Resumo ───────────────────────────────────────────────────
print(f"\n{'Base':<22}  {'Linhas':>8}  {'Cols':>5}  {'IDs':>6}  {'Turnover':>10}")
print(f"{'─'*56}")
for nome, df_base in [
    ("base_features_tt",  df_modelo),
    ("base_features_val", df_val),
    ("base_features_apl", df_apl),
]:
    n_ids = df_base[COL_ID].nunique()
    if COL_TARGET in df_base.columns and df_base[COL_TARGET].notna().any():
        tv = f"{df_base[df_base[COL_TARGET].notna()][COL_TARGET].mean():.2%}"
    else:
        tv = "N/A (apl)"
    print(f"{nome:<22}  {len(df_base):>8,}  {df_base.shape[1]:>5}  {n_ids:>6,}  {tv:>10}")

print(f"\n✓ base_features_val.parquet salva  ({path_val.stat().st_size/1e6:.1f} MB)")
print(f"✓ base_features_apl.parquet salva  ({path_apl.stat().st_size/1e6:.1f} MB)")


Transformando base_val_raw e base_apl_raw com parâmetros de TT...


C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2538751801.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_t[f"{col}_bin"] = result
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2538751801.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_t[f"{col}_bin"] = result
C:\Users\rafae\AppData\Local\Temp\ipykernel_31580\2538751801.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc


Base                      Linhas   Cols     IDs    Turnover
────────────────────────────────────────────────────────
base_features_tt          97,867    859  10,897      16.97%
base_features_val         14,223    859   5,406      18.01%
base_features_apl         20,343    859   6,179     100.00%

✓ base_features_val.parquet salva  (21.9 MB)
✓ base_features_apl.parquet salva  (30.2 MB)
